# Did the influencer ad cause purchases? Matched pairs

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Propensity Score Matching** · a worked example from the StatNex teaching guide

**The story.** A fashion brand paid an influencer to post an ad. Among 400 people, those who saw it bought more, but heavy social-media users and people who follow fashion accounts were both more likely to see the ad and more likely to buy anyway.

**The question.** How much of the extra buying is really caused by the ad?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `12_utility/propensity_score_matching.csv` (400 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Propensity score,Saw influencer ad,Heavy social media user,Follows fashion accounts,Gender,Bought product
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,1.0
0.372902,1.0,0.0,1.0,1.0,1.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.694681,0.0,1.0,1.0,0.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.189975,1.0,0.0,0.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.694681,0.0,1.0,1.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,1.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.694681,0.0,1.0,1.0,0.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.372902,1.0,0.0,1.0,1.0,1.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.189975,1.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,1.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.686409,0.0,1.0,1.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.694681,1.0,1.0,1.0,0.0,1.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.694681,0.0,1.0,1.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.196003,1.0,0.0,0.0,0.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.372902,1.0,0.0,1.0,1.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.463315,0.0,1.0,0.0,1.0,0.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.463315,1.0,1.0,0.0,1.0,1.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.686409,1.0,1.0,1.0,1.0,0.0
0.463315,1.0,1.0,0.0,1.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,1.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.463315,0.0,1.0,0.0,1.0,1.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.686409,1.0,1.0,1.0,1.0,1.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.381998,1.0,0.0,1.0,0.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,1.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,1.0
0.686409,0.0,1.0,1.0,1.0,1.0
0.381998,0.0,0.0,1.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.694681,0.0,1.0,1.0,0.0,0.0
0.372902,0.0,0.0,1.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.196003,0.0,0.0,0.0,0.0,0.0
0.189975,0.0,0.0,0.0,1.0,0.0
0.694681,1.0,1.0,1.0,0.0,0.0
0.694681,0.0,1.0,1.0,0.0,1.0
0.686409,0.0,1.0,1.0,1.0,0.0
0.472953,1.0,1.0,0.0,0.0,0.0
0.472953,0.0,1.0,0.0,0.0,0.0

These cells save `propensity_matching.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile propensity_matching.py
"""Greedy 1:1 nearest-neighbour matching on the propensity score.

Neither SciPy nor statsmodels has a matching routine, so the loop is written out here.
It is the app's: one pass, no replacement, each case taking the nearest control still
free, ties going to the control that comes first in the data.
"""

import math

import numpy as np


def greedy_match(scores, queue, controls, caliper=None):
    """Pair each case in ``queue``, in that order, with the nearest unused control.

    scores    the (possibly logit-transformed) propensity score of every case
    queue     positions of the treated cases, in the order they choose
    controls  positions of the control cases that may be chosen
    caliper   largest distance allowed for a pair; None for no limit. A treated case
              whose nearest free control is farther away stays unmatched, and that
              control stays free for the cases after it.

    Returns a list of (treated position, control position) pairs.
    """
    candidates = scores[controls]
    used = np.zeros(len(controls), dtype=bool)
    pairs = []
    for case in queue:
        if used.all():
            break
        distances = np.abs(candidates - scores[case])
        distances[used] = np.inf
        best = int(np.argmin(distances))  # the first of equal distances
        if caliper is not None and distances[best] > caliper:
            continue
        used[best] = True
        pairs.append((int(case), int(controls[best])))
    return pairs


def paired_differences(x, y):
    """x - y with the rounding noise of the subtraction taken back out.

    1.1 - 1.0 is 0.10000000000000009 in binary floating point and 0.3 - 0.2 is
    0.09999999999999998, so equal differences would not tie in a rank test. Each
    difference is rounded to a grid of 1e-12 of the data's magnitude, as the app does.
    """
    differences = np.asarray(x, float) - np.asarray(y, float)
    scale = float(np.max(np.abs(np.concatenate([x, y]))))
    if scale == 0:
        return differences
    step = 10.0 ** (math.floor(math.log10(scale)) - 12)
    return np.round(differences / step) * step

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import special, stats
from propensity_matching import greedy_match, paired_differences

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# The propensity score (0 to 1), the assignment (1 = treated, 0 = control), the
# covariates the score was built from, and the outcome. A case is used only when all
# of them are valid. The covariates and the outcome are optional (empty lists here
# when none was chosen).
# The propensity score is each case's probability of being treated given its
# covariates, usually from a logistic regression. Two cases with the same score are
# alike in the covariates on average, so comparing them mimics a randomised trial --
# but only for the covariates that were measured. Matching cannot remove a difference
# nobody recorded ("no unmeasured confounding" is an assumption, not something the
# data can check).
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
score_name = "Propensity score"
group_name = "Saw influencer ad"
covariates = ["Heavy social media user", "Follows fashion accounts", "Gender"]
outcomes = ["Bought product"]
measured = [score_name, *covariates, *outcomes]
numbers = data[[score_name, group_name, *covariates, *outcomes]].apply(
    pd.to_numeric, errors="coerce")

## Case summary

In [ ]:
# Why cases were left out. "Several" counts the cases with more than one problem.
problems = pd.DataFrame({
    "score": ~numbers[score_name].between(0, 1),
    "assignment": ~numbers[group_name].isin([0, 1]),
    "covariate": numbers[covariates].isna().any(axis=1),
    "outcome": numbers[outcomes].isna().any(axis=1),
})
how_many = problems.sum(axis=1)
counts = [(how_many == 0).sum()] + [(problems[c] & (how_many == 1)).sum() for c in problems] + [
    (how_many > 1).sum(), len(data)]
cases = pd.DataFrame({"n": counts, "share": np.array(counts) / len(data)},
                     index=["valid", "score only", "assignment only", "covariate only",
                            "outcome only", "several", "total"])
print(cases)

values = numbers[how_many == 0].reset_index(drop=True)
raw_score = values[score_name].to_numpy(float)
group = values[group_name].to_numpy(int)
treated = np.flatnonzero(group == 1)
control = np.flatnonzero(group == 0)

## The score that is matched on

In [ ]:
# The logit log(p / (1 - p)): on this scale a fixed caliper means the same at p = 0.5
# as at p = 0.02.
scores = special.logit(raw_score)

# Common support: only cases whose score lies where both groups have cases --
# between the larger of the two minimums and the smaller of the two maximums. A case
# outside it has nobody comparable in the other group ("overlap" fails there).
low = max(scores[treated].min(), scores[control].min())
high = min(scores[treated].max(), scores[control].max())
inside = (scores >= low) & (scores <= high)

## Assignment summary

In [ ]:
assignment = pd.DataFrame(
    [[(group == 0).sum(), ((group == 0) & inside).sum()],
     [(group == 1).sum(), ((group == 1) & inside).sum()],
     [len(group), inside.sum()]],
    index=[f"{group_name}=0", f"{group_name}=1", "total"],
    columns=["all cases", "within common support"],
)
print()
print(assignment)


def describe(x):
    return [len(x), x.mean(), x.var(ddof=1), x.std(ddof=1), x.max(), x.min()]


statistic_names = ["n", "mean", "variance", "SD", "maximum", "minimum"]

## Descriptive statistics

In [ ]:
statistics = pd.DataFrame({name: describe(values[name].to_numpy(float)) for name in measured},
                          index=statistic_names)
print()
print(statistics)

## Matching

In [ ]:
# Greedy one-to-one nearest-neighbour matching without replacement: each treated case
# in turn takes the control with the closest score that is still free, and that
# control is then used up. The caliper is the largest score distance a pair may have.
# Neither SciPy nor statsmodels has a matching routine, so the loop is in
# propensity_matching.py. Among equally close controls it takes the first in the data;
# with a score that takes few values (binary covariates) such ties are common, and
# another rule -- R's MatchIt, or Stata's teffects psmatch, which keeps every tied
# control -- can give a different effect.
caliper_value = 0.2
# A blank value means the default, 0.2. (The app refuses 0 or less.)
caliper_value = 0.2 if caliper_value is None else float(caliper_value)
# The caliper is caliper_value on the score's own scale.
caliper = caliper_value
inside_treated = treated[inside[treated]]
inside_control = control[inside[control]]
# Greedy matching depends on who chooses first.
# Treated cases choose in ascending order of score (ties in data order).
queue = inside_treated[np.argsort(scores[inside_treated], kind="stable")]
pairs = greedy_match(scores, queue, inside_control, caliper)
treated_index = np.array([t for t, c in pairs])
control_index = np.array([c for t, c in pairs])
print(f"\nmatched pairs: {len(pairs)} of {len(inside_treated)} treated cases")
print(f"common support: {low:.6g} to {high:.6g}")


def side_by_side(control_rows, treated_rows):
    """The statistics of each variable, control and treated next to each other."""
    table = {}
    for name in measured:
        column = values[name].to_numpy(float)
        table[(name, f"{group_name}=0")] = describe(column[control_rows])
        table[(name, f"{group_name}=1")] = describe(column[treated_rows])
    return pd.DataFrame(table, index=statistic_names)

## Before matching

In [ ]:
before = side_by_side(control, treated)
print()
print(before)

## After matching

In [ ]:
after = side_by_side(control_index, treated_index)
print()
print(after)

## Effect on the treated

In [ ]:
# The mean of (treated - control) over the matched pairs: the average effect on the
# treated (ATT), for the kind of treated cases that found a partner. Unmatched cases
# are not part of it, so if many were dropped it no longer describes the whole treated
# group. It is a causal effect only if the covariates capture every reason that
# drives both treatment and outcome. The app prints no interval for it.
outcome_name = outcomes[0]
outcome = values[outcome_name].to_numpy(float)
effect = (outcome[treated_index] - outcome[control_index]).mean()
print(f"\nmean difference in {outcome_name}: {effect:.6g}")

## Matching balance

In [ ]:
# Each covariate compared at three stages: all cases, the common support, and the
# matched pairs. The standardised difference d -- the difference in means divided by
# sqrt((SD_treated^2 + SD_control^2) / 2) -- is the one to read, because unlike a P value
# it does not depend on the sample size; |d| < 0.1 is a common rule of thumb for balance,
# not a law. Here each stage uses its own SDs; some packages (R's MatchIt summary, the
# cobalt package) divide by the SD before matching instead, so their matched d can
# differ a little. The variance ratio (treated over control) should be near 1.


def standardised(a, b):
    spread = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return 0.0 if spread == 0 else (a.mean() - b.mean()) / spread


def variance_ratio(a, b):
    return np.nan if b.var(ddof=1) == 0 else a.var(ddof=1) / b.var(ddof=1)


stages = {}
for name in covariates:
    column = values[name].to_numpy(float)
    stages[name] = [(column[group == 1], column[group == 0]),
                    (column[(group == 1) & inside], column[(group == 0) & inside]),
                    (column[treated_index], column[control_index])]
stage_names = ["all cases", "common support", "matched"]
rows, labels = [], []
for i, stage in enumerate(stage_names):
    rows.append([stages[n][i][0].mean() - stages[n][i][1].mean() for n in covariates])
    labels.append(("mean difference", stage))
for i, stage in enumerate(stage_names):
    rows.append([standardised(*stages[n][i]) for n in covariates])
    labels.append(("standardised difference", stage))
for i, stage in enumerate(stage_names[1:], start=1):
    # Fall in |d| x 100: 100 x (|d| before - |d| at this stage), the fall in the
    # standardised bias in percentage points. It is not the percent reduction
    # 100 x (1 - |d| now / |d| before) that Stata's pstest prints.
    rows.append([100 * (abs(standardised(*stages[n][0])) - abs(standardised(*stages[n][i])))
                 for n in covariates])
    labels.append(("fall in |d| x 100", stage))
for i, stage in enumerate(stage_names):
    rows.append([variance_ratio(*stages[n][i]) for n in covariates])
    labels.append(("variance ratio", stage))
balance = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(labels), columns=covariates)
print()
print(balance)

# The standardised differences again, one row per covariate, for a chart; the last
# column is the covariate's position on the chart.
chart_data = pd.DataFrame(
    [[standardised(*stages[n][0]), standardised(*stages[n][1]), standardised(*stages[n][2]),
      2.5 - i] for i, n in enumerate(covariates)],
    index=covariates, columns=["all cases", "common support", "matched", "position"])
print()
print(chart_data)

## Tests

In [ ]:
# P values are printed because readers look for them, but they are not how balance is
# judged: matching shrinks the sample, so P rises even when balance gets worse.
# Welch's t test compares the means without assuming equal variances. The
# Brunner-Munzel test asks whether a value from one group tends to be larger than one
# from the other, without assuming equal shapes; it has no value when the two groups
# do not overlap at all.


def brunner_munzel(a, b):
    try:
        p = stats.brunnermunzel(a, b).pvalue
    except (ValueError, ZeroDivisionError):
        return np.nan
    return p if np.isfinite(p) else np.nan


tests = {}
tests[("common support", "Welch")] = [
    stats.ttest_ind(*stages[n][1], equal_var=False).pvalue for n in covariates]
tests[("common support", "Brunner-Munzel")] = [brunner_munzel(*stages[n][1]) for n in covariates]
# After matching, the pairs are compared as pairs: the paired t test and Wilcoxon's
# signed-rank test. When every pair has the same difference neither can be computed.
paired_t, signed_rank = [], []
for n in covariates:
    treated_values, control_values = stages[n][2]
    difference = paired_differences(treated_values, control_values)
    if difference.var(ddof=1) == 0:
        paired_t.append(np.nan)
        signed_rank.append(np.nan)
        continue
    paired_t.append(stats.ttest_rel(treated_values, control_values).pvalue)
    try:
        signed_rank.append(stats.wilcoxon(difference).pvalue)
    except ValueError:
        signed_rank.append(np.nan)
tests[("matched", "paired t")] = paired_t
tests[("matched", "Wilcoxon signed-rank")] = signed_rank
tests = pd.DataFrame(tests, index=covariates).T
print()
print(tests)
print("(NaN: not computable -- every pair has the same difference)")

## Matched pairs

In [ ]:
# Row i of each table is one pair, in the order the pairs were formed.
pairs_control = values.loc[control_index, measured].assign(**{group_name: 0})
pairs_control = pairs_control[[score_name, group_name, *covariates, *outcomes]]
pairs_control.index = pd.RangeIndex(1, len(pairs) + 1, name="No.")
pairs_treated = values.loc[treated_index, measured].assign(**{group_name: 1})
pairs_treated = pairs_treated[[score_name, group_name, *covariates, *outcomes]]
pairs_treated.index = pairs_control.index
print()
print(pd.concat({f"{group_name}=0": pairs_control, f"{group_name}=1": pairs_treated}, axis=1))

## Distributions before and after matching

In [ ]:
# Kernel density of each variable: both groups in all cases, and both groups among the
# matched pairs. The matching has worked for a variable when the two matched curves
# lie on each other. Saved to figures/.
import os

import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)
for number, name in enumerate(measured, start=1):
    column = values[name].to_numpy(float)
    spread = column.std(ddof=1)
    if spread == 0:
        continue
    grid = np.linspace(column.min() - 3 * spread, column.max() + 3 * spread, 51)
    figure, axis = plt.subplots(figsize=(6, 4))
    for label, chosen in (("all, T=0", column[group == 0]), ("all, T=1", column[group == 1]),
                          ("matched, T=0", column[control_index]),
                          ("matched, T=1", column[treated_index])):
        if len(chosen) > 1 and chosen.std(ddof=1) > 0:
            # Silverman's rule of thumb, R's bw.nrd0: 0.9 min(SD, IQR/1.34) n^(-1/5),
            # falling back to the SD when the IQR is 0.
            sd = chosen.std(ddof=1)
            iqr = np.subtract(*np.quantile(chosen, [0.75, 0.25]))
            bandwidth = 0.9 * (min(sd, iqr / 1.34) if iqr > 0 else sd) * len(chosen) ** -0.2
            axis.plot(grid, stats.gaussian_kde(chosen, bw_method=bandwidth / sd)(grid), label=label)
    axis.set(xlabel=name, ylabel="density")
    axis.legend()
    figure.savefig(f"figures/density_{number}.png", dpi=120, bbox_inches="tight")
    plt.close(figure)

## What the result says

Before matching, 35.9% of the 156 people who saw the ad bought, against 21.7% of the 244 who did not. But 62% of viewers are heavy social media users, against 34% of the others.

The propensity score is each person's estimated chance of seeing the ad. Matching pairs each viewer with a non-viewer who has almost the same score. It found 133 pairs. In the matched data the standardised difference in means, a measure of imbalance between the groups, falls from 0.59 to 0 for heavy users and from 0.35 to 0 for fashion followers. Gender is the exception: its standardised difference rises from 0.03 to 0.09, a small imbalance that the matching did not remove.

Among the pairs, 33.8% of viewers bought against 26.3%: a mean difference of 0.075, or 7.5 percentage points. So about half of the apparent effect was the ad reaching people who would have bought anyway. Viewers without a match are left out of this estimate.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.